# Project 28 — Multilingual Local RAG

## Retrieve in One Language, Answer in Another

**Goal:** Build a cross-lingual RAG system that indexes documents in
multiple languages and generates answers in a target language.

**Stack:** Ollama · LangChain · ChromaDB · Jupyter

### What You'll Learn
1. Cross-lingual retrieval with multilingual embeddings
2. Language-controlled answer generation
3. Retrieval quality comparison by query language

### Prerequisites
- **Ollama** with `nomic-embed-text` and `qwen3:8b`

In [ ]:
!pip install -q langchain langchain-ollama langchain-community chromadb

## Step 1 — Verify Ollama Is Running

In [ ]:
import requests
OLLAMA_BASE = "http://localhost:11434"
try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    print(f"Ollama running — {len(r.json().get('models',[]))} model(s)")
except Exception as e:
    print(f"Cannot reach Ollama: {e}")

## Step 2 — Multilingual Corpus

Documents about the same topics in English, French, German, and Spanish.

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.schema import Document
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.2)
emb = OllamaEmbeddings(model="nomic-embed-text")

docs = [
    Document(page_content="Python is a versatile programming language used for web development, "
        "data science, machine learning, and automation.",
        metadata={"lang": "en", "topic": "python"}),
    Document(page_content="Machine learning algorithms learn patterns from data to make "
        "predictions. Types include supervised, unsupervised, and reinforcement learning.",
        metadata={"lang": "en", "topic": "ml"}),
    Document(page_content="Climate change is caused by greenhouse gas emissions from burning "
        "fossil fuels, deforestation, and industrial processes.",
        metadata={"lang": "en", "topic": "climate"}),
    Document(page_content="Python est un langage de programmation polyvalent utilise pour le "
        "developpement web, la science des donnees et l'intelligence artificielle.",
        metadata={"lang": "fr", "topic": "python"}),
    Document(page_content="Les algorithmes d'apprentissage automatique apprennent a partir "
        "des donnees. Types: supervise, non supervise, par renforcement.",
        metadata={"lang": "fr", "topic": "ml"}),
    Document(page_content="Python ist eine vielseitige Programmiersprache fuer Webentwicklung, "
        "Datenwissenschaft, maschinelles Lernen und Automatisierung.",
        metadata={"lang": "de", "topic": "python"}),
    Document(page_content="Der Klimawandel wird durch Treibhausgasemissionen aus der Verbrennung "
        "fossiler Brennstoffe und Abholzung verursacht.",
        metadata={"lang": "de", "topic": "climate"}),
    Document(page_content="Python es un lenguaje de programacion versatil utilizado para "
        "desarrollo web, ciencia de datos e inteligencia artificial.",
        metadata={"lang": "es", "topic": "python"}),
]

vs = Chroma.from_documents(docs, emb, collection_name="multilingual")
print(f"Indexed {len(docs)} docs in {len(set(d.metadata['lang'] for d in docs))} languages")

## Step 3 — Cross-Lingual Retrieval

In [ ]:
for q, ql in [("What is Python?", "en"), ("Qu'est-ce que Python?", "fr"),
              ("Was ist Python?", "de"), ("Que es Python?", "es")]:
    results = vs.similarity_search_with_score(q, k=4)
    print(f"\n[{ql}] {q}")
    for doc, sc in results:
        print(f"  [{doc.metadata['lang']}] {sc:.4f} {doc.metadata['topic']} — "
              f"{doc.page_content[:40]}...")

## Step 4 — Answer in Target Language

In [ ]:
tp = ChatPromptTemplate.from_messages([
    ("system", "Answer using the context. IMPORTANT: Respond in {target_lang}."),
    ("human", "Context: {context}\n\nQuestion: {question}\n\nAnswer in {target_lang}:")
])
tc = tp | llm | StrOutputParser()

retrieved = vs.similarity_search("What is Python?", k=3)
ctx = "\n".join([d.page_content for d in retrieved])

for lang in ["English", "French", "German", "Spanish"]:
    ans = tc.invoke({"context": ctx, "question": "What is Python?", "target_lang": lang})
    print(f"  [{lang[:2].upper()}] {str(ans)[:120]}")

## Step 5 — Retrieval Quality by Language

In [ ]:
print(f"{'Query Language':<16} {'Correct topic':>14}")
print("-" * 32)
for ql, q, exp in [
    ("English", "What is Python?", "python"),
    ("French", "Qu'est-ce que Python?", "python"),
    ("German", "Was ist Python?", "python"),
    ("Spanish", "Que es Python?", "python"),
    ("English", "What causes climate change?", "climate"),
    ("German", "Was verursacht Klimawandel?", "climate"),
]:
    r = vs.similarity_search(q, k=1)
    hit = "Y" if r[0].metadata["topic"] == exp else "N"
    print(f"  {ql:<14} {hit} (got: {r[0].metadata['topic']})")

## Limitations

1. `nomic-embed-text` is primarily English-focused; dedicated multilingual
   models would improve cross-lingual retrieval.
2. Translation quality depends on the LLM's multilingual capability.
3. Small corpus — cross-lingual patterns more visible at scale.

## What You Learned

| Concept | What We Did |
|---|---|
| **Multilingual corpus** | EN / FR / DE / ES documents |
| **Cross-lingual retrieval** | Query in one language, retrieve from all |
| **Language-controlled generation** | Answer in any target language |
| **Quality analysis** | Compare retrieval across languages |